# 04 — Robot Parameter Analysis

Önce baseline sonuçlarını doğrular, ardından yalnızca geliştirme döneminde kontrollü bir sinyal parametresi taraması yaparız. Holdout dönemi parametre seçiminde kullanılmaz.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = (
    Path.cwd()
    if (Path.cwd() / "src").exists()
    else Path.cwd().parent
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.config import StrategyConfig, PortfolioConfig
from src.features import add_indicators
from src.signals import build_market_regime, add_robot_scores
from src.backtest import run_portfolio_backtest
from src.metrics import (
    portfolio_metrics,
    yearly_performance,
)
from src.experiments import (
    evaluate_strategy,
    run_strategy_grid,
    apply_robustness_filters,
    compare_periods,
)

sns.set_theme(style="whitegrid")


c:\Users\okand\Desktop\Projects\Algorithmic Trading\BIST-Algo-Trade\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Verileri ve özellikleri hazırla


In [2]:
stock_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bist100_robot_clean.parquet"
)

market_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xu100_robot_clean.parquet"
)

stock_features = add_indicators(stock_prices)
market_features = add_indicators(market_prices)
market_regime = build_market_regime(market_features)

print("Hisse özelliği:", stock_features.shape)
print("Hisse sayısı:", stock_features["Ticker"].nunique())
print(
    "Tarih aralığı:",
    stock_features["Date"].min(),
    "→",
    stock_features["Date"].max(),
)


Hisse özelliği: (152459, 42)
Hisse sayısı: 99
Tarih aralığı: 2018-10-18 00:00:00 → 2026-07-24 00:00:00


## 2. Baseline sonucu yeniden üret ve doğrula


In [3]:
base_strategy = StrategyConfig()
base_portfolio = PortfolioConfig()

baseline_scored = add_robot_scores(
    stock_features,
    market_regime,
    base_strategy,
    include_reasons=False,
)

baseline_equity, baseline_trades = run_portfolio_backtest(
    baseline_scored,
    base_strategy,
    base_portfolio,
)

baseline_metrics = portfolio_metrics(
    baseline_equity,
    baseline_trades,
)

display(pd.DataFrame([baseline_metrics]))
display(yearly_performance(baseline_equity))


,Start_Value,End_Value,Total_Return_%,CAGR_%,Max_Drawdown_%,Profit_Factor,Win_Rate_%,Expectancy_%,Sharpe,Sortino,Calmar,Trade_Count,Outlier_Count,Exposure_%,Average_Open_Positions,Max_Open_Positions,Average_Invested_%
0,500000.0,9.435985e+06,1787.197014,45.986671,-27.915671,2.054777,37.621359,5.219497,2.037607,2.681224,1.647342,824,0,98.557445,7.25863,8,68.100607


,Year,Start_Equity,End_Equity,Return_%,Yearly_Max_DD_%
0,2018,5.000000e+05,4.825094e+05,-3.498118,-3.689719
1,2019,4.801923e+05,6.512790e+05,35.628783,-12.633757
2,2020,6.552528e+05,1.053568e+06,60.788089,-13.292287
3,2021,1.065350e+06,1.134853e+06,6.523961,-27.915671
4,2022,1.150792e+06,2.846321e+06,147.335903,-7.997293
5,2023,2.920930e+06,3.393362e+06,16.174043,-22.140374
6,2024,3.393370e+06,5.362762e+06,58.036482,-8.889749
7,2025,5.357448e+06,7.877723e+06,47.042445,-8.436909
8,2026,7.956814e+06,9.435985e+06,18.589987,-15.905753


In [4]:
assert not baseline_equity.empty
assert baseline_equity["Equity"].gt(0).all()
assert baseline_equity["Open_Positions"].le(
    base_portfolio.max_positions
).all()

if not baseline_trades.empty:
    assert baseline_trades["Shares"].gt(0).all()
    assert (
        baseline_trades["Exit_Date"]
        >= baseline_trades["Entry_Date"]
    ).all()

print(
    "Outlier işlem:",
    int(
        baseline_trades.get(
            "Is_Outlier",
            pd.Series(dtype=bool),
        ).sum()
    ),
)

display(
    baseline_trades.nsmallest(
        10,
        "Return_%",
    )
)

display(
    baseline_trades.nlargest(
        10,
        "Return_%",
    )
)


Outlier işlem: 0


,Ticker,Entry_Date,Exit_Date,Entry,Exit,Shares,Return,Return_%,Reason,Is_Outlier,Signal_Score
120,GSRAY.IS,2020-02-21,2020-03-12,0.542939,0.381615,47347,-0.299938,-29.993752,LOW10 Altı,False,12
109,SASA.IS,2020-02-11,2020-02-28,0.189689,0.146859,280722,-0.228881,-22.888133,LOW10 Altı,False,12
431,KUYAS.IS,2023-01-17,2023-02-08,6.558090,5.199580,20115,-0.210315,-21.031532,LOW10 Altı,False,11
111,TRMET.IS,2020-02-25,2020-02-28,12.645240,10.079800,4848,-0.206060,-20.606001,Stop Loss Gap,False,14
520,MIATK.IS,2023-11-02,2023-11-21,55.564754,46.063657,2762,-0.174301,-17.430084,Stop Loss,False,12
810,DSTKF.IS,2026-06-25,2026-07-16,3336.660000,2796.895000,245,-0.165114,-16.511429,LOW10 Altı,False,12
315,TKFEN.IS,2021-12-20,2021-12-22,21.195980,17.856693,3870,-0.160907,-16.090652,Stop Loss Gap,False,12
530,BSOKE.IS,2023-12-07,2023-12-12,3.466920,2.928235,47120,-0.158750,-15.875016,Stop Loss,False,12
814,SKBNK.IS,2026-07-02,2026-07-22,16.933800,14.480980,47873,-0.148261,-14.826133,LOW10 Altı,False,12
666,GENIL.IS,2025-04-09,2025-04-10,11.328122,9.690019,31423,-0.148020,-14.801972,Stop Loss Gap,False,12


,Ticker,Entry_Date,Exit_Date,Entry,Exit,Shares,Return,Return_%,Reason,Is_Outlier,Signal_Score
501,MIATK.IS,2023-07-26,2023-09-28,14.159030,44.257461,13146,2.113263,211.326295,Trailing Stop,False,14
221,ISMEN.IS,2020-08-05,2021-03-09,1.168525,3.217357,57669,1.742357,174.235729,Trailing Stop,False,12
156,FENER.IS,2020-08-11,2020-09-11,2.905800,7.964040,14752,1.729798,172.979824,Trailing Stop,False,14
601,BSOKE.IS,2024-06-13,2024-09-18,7.590150,20.583750,41995,1.701077,170.107704,Trailing Stop,False,14
706,QUAGR.IS,2025-03-20,2025-09-29,3.877740,10.179600,117526,1.614658,161.465776,Trailing Stop,False,12
395,HEKTS.IS,2022-09-30,2022-11-21,15.732254,37.325202,6966,1.363056,136.305608,Trailing Stop,False,12
386,ODAS.IS,2022-07-19,2022-10-19,3.507000,8.123720,33420,1.307183,130.718287,Trailing Stop,False,14
786,KTLEV.IS,2026-03-05,2026-05-22,47.471581,109.862912,11425,1.305049,130.504936,Trailing Stop,False,11
670,TUREX.IS,2025-03-20,2025-05-12,18.537000,41.417000,23180,1.225369,122.536887,Trailing Stop,False,12
775,DSTKF.IS,2026-02-20,2026-04-30,1170.336000,2594.800000,631,1.208290,120.829020,Trailing Stop,False,12


## 3. Zaman ayrımı

- Geliştirme: 2018–2022
- Doğrulama: 2023–2024
- Holdout: 2025–son veri

Holdout dönemi yalnızca strateji seçildikten sonra bir kez çalıştırılmalıdır.


In [5]:
PERIODS = {
    "Development": ("2018-01-01", "2022-12-31"),
    "Validation": ("2023-01-01", "2024-12-31"),
    "Holdout": (
        "2025-01-01",
        stock_features["Date"].max().strftime("%Y-%m-%d"),
    ),
}

baseline_period_records = []

for period_name, (start, end) in PERIODS.items():
    metrics, _, _ = evaluate_strategy(
        stock_features=stock_features,
        market_regime=market_regime,
        strategy_config=base_strategy,
        portfolio_config=base_portfolio,
        start=start,
        end=end,
    )

    metrics["Period"] = period_name
    baseline_period_records.append(metrics)

baseline_period_results = pd.DataFrame(
    baseline_period_records
)

display(
    baseline_period_results[
        [
            "Period",
            "CAGR_%",
            "Max_Drawdown_%",
            "Profit_Factor",
            "Win_Rate_%",
            "Sharpe",
            "Calmar",
            "Trade_Count",
        ]
    ]
)


,Period,CAGR_%,Max_Drawdown_%,Profit_Factor,Win_Rate_%,Sharpe,Calmar,Trade_Count
0,Development,51.203917,-27.915671,2.331645,40.669856,2.237308,1.834236,418
1,Validation,34.900812,-22.595839,1.732578,33.187773,1.615362,1.544568,229
2,Holdout,38.115150,-15.877328,1.734218,34.848485,1.760896,2.400602,198


## 4. Aşama 1 — Sinyal filtresi taraması

İlk aşamada stop ve portföy riskini sabit tutuyoruz. Yalnızca skor, ADX ve hacim eşiğini test ediyoruz.


In [6]:
signal_grid = {
    "buy_score": [10, 11, 12],
    "minimum_adx": [18.0, 20.0, 22.0],
    "volume_multiplier": [1.10, 1.30, 1.50],
}

development_results = run_strategy_grid(
    stock_features=stock_features,
    market_regime=market_regime,
    base_strategy=base_strategy,
    portfolio_config=base_portfolio,
    parameter_grid=signal_grid,
    start=PERIODS["Development"][0],
    end=PERIODS["Development"][1],
)

development_results.to_csv(
    PROJECT_ROOT
    / "results"
    / "signal_grid_development.csv",
    index=False,
)

print("Deney sayısı:", len(development_results))
display(
    development_results[
        development_results["Status"].eq("OK")
    ].sort_values(
        ["Calmar", "CAGR_%"],
        ascending=False,
    ).head(15)
)


Strateji grid: 2018-01-01 → 2022-12-31: 100%|██████████| 27/27 [03:10<00:00,  7.06s/it]

Deney sayısı: 27


,Start_Value,End_Value,Total_Return_%,CAGR_%,Max_Drawdown_%,Profit_Factor,Win_Rate_%,Expectancy_%,Sharpe,Sortino,...,Portfolio_risk_per_trade,Portfolio_max_positions,Portfolio_commission_rate,Portfolio_slippage_rate,Portfolio_minimum_valid_return,Portfolio_maximum_valid_return,Portfolio_max_position_fraction,Experiment_ID,Status,Error
19,500000.0,3.693504e+06,638.700768,60.984835,-21.036353,2.639109,42.929293,7.036945,2.580370,3.192570,...,0.0075,8,0.002,0.002,-0.8,5.0,None,20,OK,
25,500000.0,3.697316e+06,639.463207,61.024382,-21.171012,2.683027,43.701799,7.158419,2.570991,3.186590,...,0.0075,8,0.002,0.002,-0.8,5.0,None,26,OK,
8,500000.0,4.533746e+06,806.749245,69.036532,-24.094333,2.569436,43.115124,6.974171,2.688378,3.423797,...,0.0075,8,0.002,0.002,-0.8,5.0,None,9,OK,
2,500000.0,3.967876e+06,693.575121,63.755016,-23.185612,2.501705,43.150685,6.519804,2.579538,3.225796,...,0.0075,8,0.002,0.002,-0.8,5.0,None,3,OK,
23,500000.0,3.785323e+06,657.064619,61.928835,-22.640914,2.747399,43.799472,7.621581,2.600752,3.177687,...,0.0075,8,0.002,0.002,-0.8,5.0,None,24,OK,
5,500000.0,3.749443e+06,649.888517,61.562044,-22.518699,2.454501,42.307692,6.406282,2.520344,3.174206,...,0.0075,8,0.002,0.002,-0.8,5.0,None,6,OK,
3,500000.0,3.281480e+06,556.295913,56.514252,-21.040440,2.384741,42.562929,6.099785,2.399292,2.961439,...,0.0075,8,0.002,0.002,-0.8,5.0,None,4,OK,
22,500000.0,3.555479e+06,611.095837,59.531572,-22.471708,2.696930,42.635659,7.373018,2.546241,3.178787,...,0.0075,8,0.002,0.002,-0.8,5.0,None,23,OK,
17,500000.0,3.641300e+06,628.260056,60.440128,-23.049289,2.540772,43.192488,6.672825,2.464462,3.046338,...,0.0075,8,0.002,0.002,-0.8,5.0,None,18,OK,
6,500000.0,3.453642e+06,590.728300,58.431514,-22.665699,2.417265,41.935484,6.330896,2.434949,3.067114,...,0.0075,8,0.002,0.002,-0.8,5.0,None,7,OK,


In [7]:
robust_development = apply_robustness_filters(
    development_results,
    minimum_trades=40,
    maximum_drawdown_limit=-35.0,
    minimum_profit_factor=1.10,
)

selected_development = robust_development.head(5).copy()

display(
    selected_development[
        [
            "Experiment_ID",
            "Strategy_buy_score",
            "Strategy_minimum_adx",
            "Strategy_volume_multiplier",
            "CAGR_%",
            "Max_Drawdown_%",
            "Profit_Factor",
            "Sharpe",
            "Calmar",
            "Trade_Count",
        ]
    ]
)


,Experiment_ID,Strategy_buy_score,Strategy_minimum_adx,Strategy_volume_multiplier,CAGR_%,Max_Drawdown_%,Profit_Factor,Sharpe,Calmar,Trade_Count
0,20,12,18.0,1.3,60.984835,-21.036353,2.639109,2.580370,2.899021,396
1,26,12,22.0,1.3,61.024382,-21.171012,2.683027,2.570991,2.882450,389
2,9,10,22.0,1.5,69.036532,-24.094333,2.569436,2.688378,2.865260,443
3,3,10,18.0,1.5,63.755016,-23.185612,2.501705,2.579538,2.749766,438
4,24,12,20.0,1.5,61.928835,-22.640914,2.747399,2.600752,2.735262,379


## 5. İlk beş konfigürasyonu doğrulama döneminde karşılaştır


In [8]:
parameter_columns = [
    "Strategy_buy_score",
    "Strategy_minimum_adx",
    "Strategy_volume_multiplier",
]

selected_comparison = compare_periods(
    stock_features=stock_features,
    market_regime=market_regime,
    configurations=selected_development,
    base_strategy=base_strategy,
    portfolio_config=base_portfolio,
    periods={
        "Development": PERIODS["Development"],
        "Validation": PERIODS["Validation"],
    },
    parameter_columns=parameter_columns,
)

comparison_view = selected_comparison[
    [
        "Selected_Config",
        "Period_Name",
        "Strategy_buy_score",
        "Strategy_minimum_adx",
        "Strategy_volume_multiplier",
        "CAGR_%",
        "Max_Drawdown_%",
        "Profit_Factor",
        "Sharpe",
        "Calmar",
        "Trade_Count",
    ]
].sort_values(
    ["Selected_Config", "Period_Name"]
)

display(comparison_view)

selected_comparison.to_csv(
    PROJECT_ROOT
    / "results"
    / "signal_grid_selected_validation.csv",
    index=False,
)


,Selected_Config,Period_Name,Strategy_buy_score,Strategy_minimum_adx,Strategy_volume_multiplier,CAGR_%,Max_Drawdown_%,Profit_Factor,Sharpe,Calmar,Trade_Count
0,1,Development,12,18.0,1.3,60.984835,-21.036353,2.639109,2.580370,2.899021,396
1,1,Validation,12,18.0,1.3,27.310294,-18.310984,1.626837,1.385437,1.491471,215
2,2,Development,12,22.0,1.3,61.024382,-21.171012,2.683027,2.570991,2.882450,389
3,2,Validation,12,22.0,1.3,26.058562,-17.395542,1.610119,1.339142,1.498002,215
4,3,Development,10,22.0,1.5,69.036532,-24.094333,2.569436,2.688378,2.865260,443
5,3,Validation,10,22.0,1.5,28.699341,-24.683760,1.539181,1.372094,1.162681,249
6,4,Development,10,18.0,1.5,63.755016,-23.185612,2.501705,2.579538,2.749766,438
7,4,Validation,10,18.0,1.5,25.208648,-25.821817,1.421071,1.269692,0.976254,247
8,5,Development,12,20.0,1.5,61.928835,-22.640914,2.747399,2.600752,2.735262,379
9,5,Validation,12,20.0,1.5,25.381482,-22.017760,1.568790,1.319810,1.152773,221


Bu aşamada holdout sonucu üzerinden seçim yapma. Geliştirme ve doğrulama dönemlerinin ikisinde de dengeli kalan konfigürasyonu seçtikten sonra çıkış parametreleri test edilecektir.


In [9]:
baseline_period_results[
    [
        "Period",
        "CAGR_%",
        "Max_Drawdown_%",
        "Profit_Factor",
        "Win_Rate_%",
        "Sharpe",
        "Calmar",
        "Trade_Count",
    ]
]

,Period,CAGR_%,Max_Drawdown_%,Profit_Factor,Win_Rate_%,Sharpe,Calmar,Trade_Count
0,Development,51.203917,-27.915671,2.331645,40.669856,2.237308,1.834236,418
1,Validation,34.900812,-22.595839,1.732578,33.187773,1.615362,1.544568,229
2,Holdout,38.115150,-15.877328,1.734218,34.848485,1.760896,2.400602,198


In [10]:
comparison_view

,Selected_Config,Period_Name,Strategy_buy_score,Strategy_minimum_adx,Strategy_volume_multiplier,CAGR_%,Max_Drawdown_%,Profit_Factor,Sharpe,Calmar,Trade_Count
0,1,Development,12,18.0,1.3,60.984835,-21.036353,2.639109,2.580370,2.899021,396
1,1,Validation,12,18.0,1.3,27.310294,-18.310984,1.626837,1.385437,1.491471,215
2,2,Development,12,22.0,1.3,61.024382,-21.171012,2.683027,2.570991,2.882450,389
3,2,Validation,12,22.0,1.3,26.058562,-17.395542,1.610119,1.339142,1.498002,215
4,3,Development,10,22.0,1.5,69.036532,-24.094333,2.569436,2.688378,2.865260,443
5,3,Validation,10,22.0,1.5,28.699341,-24.683760,1.539181,1.372094,1.162681,249
6,4,Development,10,18.0,1.5,63.755016,-23.185612,2.501705,2.579538,2.749766,438
7,4,Validation,10,18.0,1.5,25.208648,-25.821817,1.421071,1.269692,0.976254,247
8,5,Development,12,20.0,1.5,61.928835,-22.640914,2.747399,2.600752,2.735262,379
9,5,Validation,12,20.0,1.5,25.381482,-22.017760,1.568790,1.319810,1.152773,221
